### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:

- **`numpy` (المستوردة كـ `np`):** للعمليات الحسابية والتعامل مع المصفوفات الرياضية.
- **`pandas` (المستوردة كـ `pd`):** لقراءة البيانات وإدارة الجداول البرمجية (DataFrames).
- **`matplotlib.pyplot` (المستوردة كـ `plt`):** للرسم البياني وتصور البيانات بصرياً.
- **`seaborn` (المستوردة كـ `sns`):** لإنشاء رسومات بيانية إحصائية جذابة ومتقدمة.
- **`train_test_split`:** لتقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار بشكل عشوائي ومنظم.
- **`StandardScaler`:** لتقييس وتوحيد نطاق الخصائص (Feature Scaling) ليكون المتوسط صفر والانحراف المعياري واحد.
- **`LinearRegression`:** لبناء وتدريب نموذج الانحدار الخطي (البسيط أو المتعدد).
- **`statsmodels.api` (المستوردة كـ `sm`):** لإجراء التحليلات الإحصائية المتقدمة والحصول على تقرير مفصل للنموذج (OLS Summary).


In [ ]:
# Cell 1: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:




In [ ]:
# Cell 2: Load dataset
import os
import urllib.request

filename = '50_Startups.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/2-%20Multi-Linear%20Regression/50_Startups.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset


### ثالثاً: استكشاف البيانات بصرياً وإحصائياً (Exploratory Data Analysis - EDA)
نقوم بتحليل البيانات لمعرفة العلاقات والارتباط بين المتغيرات المختلفة:

- **`pairplot`:** لرسم العلاقات الثنائية وتوزيع البيانات لكل الأعمدة.
- يساعدنا هذا التحليل البصري في فهم العلاقات الخطية أو غير الخطية بين المتغيرات قبل إدخالها للنموذج.


In [ ]:
# Cell 3: Visual exploratory analysis
sns.pairplot(dataset)
plt.show()


### رابعاً: معالجة البيانات الفئوية (Encoding Categorical Data)
الخوارزميات الرياضية تتعامل فقط مع الأرقام، لذلك نقوم بترميز النصوص أو الفئات (Categorical Data):
- نستخدم تقنية الترميز الأحادي (One-Hot Encoding) مثل `pd.get_dummies` لتحويل الأعمدة النصية إلى أعمدة رقمية (0 و 1).
- نستخدم `drop_first=True` لتجنب مشكلة التعدد الخطي العشوائي (Dummy Variable Trap).


In [ ]:
# Cell 4: One-hot encode categorical feature State
dataset = pd.get_dummies(dataset, columns=['State'], drop_first=True)
dataset


### سادساً: اكتشاف القيم الشاذة (Outliers Detection)
نقوم بفحص البيانات للبحث عن أي قيم شاذة (Outliers) قد تؤثر سلباً على تدريب النموذج ودقته:
- نستخدم طريقة النطاق الربيعي (IQR - Interquartile Range) لحساب الحدود الدنيا والعليا للبيانات الطبيعية.
- أي قيمة تقع خارج هذه الحدود تعتبر شاذة ويتم رصدها لاتخاذ القرار المناسب بشأنها.


In [ ]:
# Cell 5: Detect outliers using IQR and boxplots
Q1 = dataset['Profit'].quantile(0.25)
Q3 = dataset['Profit'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = dataset[(dataset['Profit'] < lower_bound) | (dataset['Profit'] > upper_bound)]
print("Outliers detected:")
print(outliers)


### ثالثاً: استكشاف البيانات بصرياً وإحصائياً (Exploratory Data Analysis - EDA)
نقوم بتحليل البيانات لمعرفة العلاقات والارتباط بين المتغيرات المختلفة:

- **`heatmap` (خريطة الارتباط):** لتوضيح معامل الارتباط بين المتغيرات رقمياً ولونياً وملاحظة أي تداخل خطي (Multicollinearity).
- يساعدنا هذا التحليل البصري في فهم العلاقات الخطية أو غير الخطية بين المتغيرات قبل إدخالها للنموذج.


In [ ]:
# Cell 6: Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(dataset.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


### خامساً: فحص التعدد الخطي باستخدام عامل تضخم التباين (VIF)
نقوم بحساب **عامل تضخم التباين (Variance Inflation Factor - VIF)** للتأكد من عدم وجود ارتباط قوي جداً بين المتغيرات مستقلة (Multicollinearity):
- قيمة VIF أكبر من 5 أو 10 تشير عادة إلى ارتباط قوي قد يضر بدقة وتفسير نموذج الانحدار الخطي المتعدد.
- نقوم بحساب هذه القيم لكل ميزة لمساعدتنا في اتخاذ قرار الإبقاء عليها أو حذفها.


In [ ]:
# Cell 7: Check multicollinearity with VIF
X_vif = dataset.drop('Profit', axis=1)
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]
vif_data


### خطوة: Cell 8: Select features and target
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# Cell 8: Select features and target
X = dataset.drop('Profit', axis=1).values
y = dataset['Profit'].values


### ثامناً: تقسيم البيانات إلى مجموعتي تدريب واختبار (Train/Test Split)
نقسم البيانات بنسبة 20% لمجموعة الاختبار وبقية البيانات لمجموعة التدريب:
- **بيانات التدريب (Training Set):** لتعليم النموذج وضبط أوزانه ومعاملاته.
- **بيانات الاختبار (Test Set):** لتقييم النموذج واختبار قدرته على التنبؤ ببيانات جديدة كلياً.


In [ ]:
# Cell 9: Split dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


### تاسعاً: تقييس الخصائص (Feature Scaling)
نقوم بعملية التقييس أو المعايرة للبيانات:
- نستخدم `fit_transform` على مجموعة التدريب ليتعلم المتوسط والانحراف المعياري ويطبق التحويل.
- نستخدم `transform` فقط على مجموعة الاختبار لمنع تسرب البيانات (Data Leakage).
- هذه الخطوة ضرورية جداً للخوارزميات الحساسة للمقاييس مثل متجهات الدعم (SVM)، الجار الأقرب (KNN)، والشبكات العصبية.


In [ ]:
# Cell 10: Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


### عاشراً: بناء وتدريب نموذج الانحدار الخطي (Linear Regression)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# Cell 11: Train Multiple Linear Regression model
regressor = LinearRegression()
regressor.fit(X_train, y_train)


### الحادي عشر: التنبؤ بقيم مجموعة الاختبار (Make Predictions)
نستخدم النموذج المدرب للتنبؤ بالنتائج للمدخلات الموجودة في مجموعة الاختبار للتأكد من قدرة النموذج على التعميم على بيانات جديدة لم يتدرب عليها من قبل.


In [ ]:
# Cell 12: Predict test set results
y_pred = regressor.predict(X_test)


### الثالث عشر: تفسير النموذج والتحليلات الإحصائية (Model Interpretation)
نقوم باستخراج المعاملات الرياضية للنموذج:
- **المعاملات (Coefficients/Slope):** توضح مدى تأثير كل متغير مستقل على المتغير التابع.
- **الجزء المقطوع (Intercept):** القيمة المتوقعة عندما تكون جميع المدخلات صفراً.
- **ملخص التحليل (Summary):** يعطي تفاصيل إحصائية شاملة (مثل قيم P-value) لمعرفة الأهمية الإحصائية لكل ميزة.


In [ ]:
# Cell 13: Model interpretation (Coefficients & Summary)
print("Coefficients:", regressor.coef_)
print("Intercept:", regressor.intercept_)
X_train_with_constant = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_with_constant).fit()
print(model.summary())


### الثاني عشر: تقييم أداء النموذج (Evaluation Metrics)
نقوم بحساب عدة مقاييس إحصائية لتقييم كفاءة ودقة النموذج المستعمل:

- **معامل التحديد ($R^2$):** يقيس جودة ملاءمة النموذج للبيانات (كلما اقترب من 1 كان أفضل).
- **متوسط مربع الخطأ (MSE) وجذره (RMSE):** يقيس حجم الخطأ في التوقعات (يفضل أن يكون أقل ما يمكن).
- **متوسط الخطأ المطلق (MAE):** متوسط الفروق المطلقة بين التوقع والواقع.


In [ ]:
# Cell 14: Calculate Performance Metrics (MAE, MSE, R^2)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"R-squared (R^2) Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")


### الرابع عشر: رسم النتائج بيانيا (Visualization of Results)
نقوم برسم النقاط الحقيقية (الحمراء عادة) والخط أو المنحنى الممثل للنموذج (الأزرق) بصرياً:
- يساعدنا الرسم البياني في التحقق البصري المباشر من مدى دقة التوقعات وملاءمة النموذج للبيانات الحقيقية.


In [ ]:
# Cell 15: Residual diagnostics plot
residuals = y_test - y_pred
plt.scatter(y_pred, residuals, color='blue')
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuals vs Fitted Values')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.show()
